# 보행망 군집의 다음 한 시간 대여 수요

전 단계의 보행망 군집에 2023~2024년의 출발·반납을 배정한다. 2024년 각 분기 직전까지 학습한 MLP와 GRU를 기본, 직전 반납량, 유사 기상, 두 피처 모두의 조건에서 비교한다. 준비물은 저장소 README의 필터 처리된 연도별 ZIP 두 개와 기상 CSV다. 첫 실행은 월별 원천을 집계한다. 이전에 검증된 월별 집계가 있으면 환경변수 `BIKE_AGGREGATE_CACHE`로 그 경로를 지정할 수 있다. Python 3.11, PyTorch, numpy, pandas, pyarrow를 사용한다.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "전처리/walking_forecast.py").is_file())
OUT = ROOT / "모델"
SITE = ROOT / "따릉이 대여소 비교"
sys.path.insert(0,str(ROOT/"전처리"))
from walking_forecast import (hours, count_departures, prepare_inputs, ClusterNet, VARIANTS,
                              DEVICE, BATCH, EPOCHS, SEED)
stations = pd.read_csv(SITE/"보행망 군집 배정.csv").set_index("대여소번호").sort_index()
assert len(stations)==stations.index.nunique()==2766
from torch import nn
import math
group = stations["보행망 군집번호"]
limit = int(stations["선택 거리(m)"].unique().item())
ids = pd.Index(sorted(group.unique()),name="군집번호")
N = len(ids)
totals, late_returns, confirmed = count_departures(group)
np.testing.assert_array_equal([totals[hours.year==y].sum() for y in (2023,2024)], [42810716,42219615])
context_at, batch_inputs, past, actual, weather_names = prepare_inputs(group,totals,late_returns,confirmed)
print(f"보행망 {limit}m, {N}개 군집 / 학습 장치 {DEVICE}")

보행망 700m, 1009개 군집 / 학습 장치 cuda


## 다음 1시간 대여량 예측의 기준선

각 군집에서 전날 같은 시간 또는 전주 같은 시간의 대여량을 그대로 예측값으로 사용한다. 당시까지 반납이 확인된 건수만 입력에 포함한다. 2024년 각 분기의 동일한 군집·시간 전체에서 비교한다. 오차 단위는 한 군집의 한 시간당 대여 건수다. 12월 군집을 사용했으므로 이 결과는 회고 비교로 해석한다.

In [2]:
scores = []
for lag in (24, 168):
    observed = totals[:-lag] - late_returns[lag][lag:]
    assert observed.min() >= 0
    dates = hours[lag:]
    for quarter in range(1,5):
        selected = (dates.year == 2024) & (dates.quarter == quarter)
        error = (totals[lag:][selected] - observed[selected]).astype("float64")
        scores.append([quarter, "전날 동일 시간" if lag == 24 else "전주 동일 시간",
                       error.size, np.abs(error).mean(), np.sqrt(np.square(error).mean())])
scores = pd.DataFrame(scores, columns=["분기","기준선","군집·시간 수","MAE","RMSE"])
assert scores.groupby("기준선")["군집·시간 수"].sum().eq((hours.year==2024).sum()*len(ids)).all()
display(scores.pivot(index="분기",columns="기준선",values=["MAE","RMSE"]).round(3))

MAE              RMSE         
기준선 전날 동일 시간 전주 동일 시간 전날 동일 시간 전주 동일 시간
분기                                     
1      2.132    2.049    4.298    3.975
2      3.526    3.345    6.933    6.491
3      3.074    3.040    6.051    5.965
4      2.655    2.526    5.528    4.997

## MLP·GRU를 네 입력 조건에서 새로 학습

한 모델이 모든 군집에서 학습하며 각 군집의 다음 한 시간 출발량을 예측한다. 기본 입력은 군집 임베딩, 반납이 확인된 직전 24시간·전주 동일 시간의 출발량, 달력과 직전 날씨다. 반납량만, 유사 기상만, 둘 다 추가한 버전도 각각 처음부터 학습한다. MLP는 시퀀스를 펼쳐서 읽고 GRU는 시간순으로 읽는다. 모든 조건에서 기존 실험과 동일한 MSE, Adam 0.001, 배치 8,192, 4에포크, 시드 42를 사용한다. 분기 경계 전에 확인될 수 있었던 학습 정답만 쓴다.

MLP는 은닉층 64·32와 ReLU를 사용하고, GRU는 과거 24시간을 은닉 크기 16으로 요약해 같은 구조의 출력층에 입력한다. 최종 활성화는 Softplus다. 네트워크와 입력 변환은 [공유 코드](../전처리/walking_forecast.py)에 있다.

In [3]:
def train_model(kind, cutoff, context, variant):
    torch.manual_seed(SEED)
    model = ClusterNet(kind, context.shape[1], len(VARIANTS[variant][1]), N).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
    loss_fn = nn.MSELoss()
    # t + 24 <= cutoff이고 전주 동일 시간 관측이 있는 모든 학습 행을 사용한다.
    n_rows = (cutoff - 23 - 168) * N
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(n_rows,device=DEVICE)
        for ix in order.split(BATCH):
            t,g,seq,historical_features = batch_inputs(ix + 168*N, variant)
            pred = model(seq,historical_features,context[t],g)
            loss = loss_fn(pred,past[t,g,23].float())
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),5.)
            optimizer.step()
    return model

@torch.inference_mode()
def evaluate(model, start, stop, context, variant):
    model.eval()
    sums = torch.zeros(2,device=DEVICE,dtype=torch.float64)
    for begin in range(start*N,stop*N,BATCH):
        flat = torch.arange(begin,min(stop*N,begin+BATCH),device=DEVICE)
        t,g,seq,historical_features = batch_inputs(flat, variant)
        err = (model(seq,historical_features,context[t],g) - actual[t,g]).double()
        sums += torch.stack([err.abs().sum(),err.square().sum()])
    n_rows = (stop-start)*N
    mae, mse = (sums/n_rows).cpu().tolist()
    assert math.isfinite(mae) and math.isfinite(mse)
    return n_rows, mae, math.sqrt(mse)

In [4]:
neural = []
for quarter, begin in enumerate(pd.date_range("2024-01-01","2024-10-01",freq="QS"),1):
    start = int(hours.searchsorted(begin))
    stop = int(hours.searchsorted(begin + pd.DateOffset(months=3)))
    context, scaling = context_at(start)
    for variant, (label, feature_names) in VARIANTS.items():
        for kind in ("MLP","GRU"):
            model = train_model(kind,start,context,variant)
            size,mae,rmse = evaluate(model,start,stop,context,variant)
            assert size == (stop-start)*N
            neural.append([quarter,kind,label,size,mae,rmse])
            print(f"{quarter}분기 {kind} {label} · MAE {mae:.4f}, RMSE {rmse:.4f}",flush=True)
            if quarter == 4:
                checkpoint = {"kind":kind,"weights":model.cpu().state_dict(),"cluster_ids":list(map(int,ids)),
                              "cluster_limit_m":limit,"context_size":context.shape[1],"weather":weather_names,"variant":variant,
                              "extra_features":feature_names,"scaling":scaling,"seed":SEED,
                              "epochs":EPOCHS,"cutoff":str(begin)}
                torch.save(checkpoint, OUT / f"walking_{kind.lower()}_{variant}.pt")
            del model
individual = pd.DataFrame(neural,columns=["분기","모델","추가 입력","군집·시간 수","MAE","RMSE"])


1분기 MLP 없음 · MAE 1.4517, RMSE 2.5509


1분기 GRU 없음 · MAE 1.4433, RMSE 2.5406


1분기 MLP 둘 다 · MAE 1.4379, RMSE 2.5168


1분기 GRU 둘 다 · MAE 1.3985, RMSE 2.4190


1분기 MLP 반납량만 · MAE 1.4161, RMSE 2.4788


1분기 GRU 반납량만 · MAE 1.4147, RMSE 2.4608


... (22 outputs left out)

4분기 MLP 반납량만 · MAE 1.7325, RMSE 2.9585


4분기 GRU 반납량만 · MAE 1.6735, RMSE 2.8661


4분기 MLP 유사 기상만 · MAE 1.7477, RMSE 3.0056


4분기 GRU 유사 기상만 · MAE 1.6925, RMSE 2.8923


## 분기·모델·피처별 성능

MAE와 RMSE의 단위는 보행망 군집 한 곳의 시간당 대여 건수다. 구획 간 비교는 공통 평가구역 노트북을 사용한다.

In [5]:
results = individual.copy()
keys = ["분기","모델","추가 입력"]
assert len(results)==32 and not results.duplicated(keys).any()
assert results.groupby("분기")["군집·시간 수"].nunique().eq(1).all()
assert np.isfinite(results[["MAE","RMSE"]]).all().all()
baseline = scores.loc[scores["기준선"]=="전주 동일 시간"].set_index("분기")
results["전주 대비 MAE 개선율(%)"] = 100*(1-results.MAE/results["분기"].map(baseline.MAE))
scores.to_csv(OUT/"보행망 군집 기준선 성능.csv",index=False,encoding="utf-8-sig")
results.to_csv(OUT/"보행망 군집 피처별 성능.csv",index=False,encoding="utf-8-sig")
order = ["없음","반납량만","유사 기상만","둘 다"]
for metric in ["MAE","RMSE"]:
    print(metric)
    display(results.pivot(index=["모델","분기"],columns="추가 입력",values=metric)[order].round(4))
display(results.loc[results["분기"]<=3].groupby(["모델","추가 입력"])[["MAE","RMSE"]].mean().round(4))


MAE


추가 입력       없음    반납량만  유사 기상만     둘 다
모델  분기                                
GRU 1   1.4433  1.4147  1.4186  1.3985
    2   2.2265  2.1357  2.1323  2.0758
    3   2.0654  1.9809  1.9679  1.9092
    4   1.7114  1.6735  1.6925  1.6778
MLP 1   1.4517  1.4161  1.4416  1.4379
    2   2.2147  2.1760  2.1274  2.1184
    3   2.0390  2.0203  1.9865  1.9361
    4   1.7221  1.7325  1.7477  1.7556

RMSE


추가 입력       없음    반납량만  유사 기상만     둘 다
모델  분기                                
GRU 1   2.5406  2.4608  2.4804  2.4190
    2   3.8168  3.6581  3.5799  3.4771
    3   3.4908  3.3595  3.3592  3.2531
    4   2.9290  2.8661  2.8923  2.8304
MLP 1   2.5509  2.4788  2.5102  2.5168
    2   3.8705  3.7620  3.6309  3.6196
    3   3.4981  3.4288  3.3726  3.3697
    4   2.9761  2.9585  3.0056  3.0068

MAE    RMSE
모델  추가 입력                 
GRU 둘 다     1.7945  3.0497
    반납량만    1.8438  3.1595
    없음      1.9117  3.2827
    유사 기상만  1.8396  3.1399
MLP 둘 다     1.8308  3.1687
    반납량만    1.8708  3.2232
    없음      1.9018  3.3065
    유사 기상만  1.8518  3.1712